In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/ai_workforce_transformation.csv')

df.head()

# Initial Inspection

The dataset contains workforce, hiring, productivity, and AI adoption metrics across multiple industries.

Objectives of this section:

- Verify dataset dimensions
- Inspect column data types
- Identify missing values
- Detect obvious data quality issues

In [ ]:
rows, cols = df.shape
print(f"Rows: {rows:,}")
print(f"Columns: {cols}")

df.info() 
df.describe()  


# Data Quality Assessment 

The dataset was evaluated for:

- Missing values
- Duplicate records
- Invalid data types
- Potential outliers

This step helps identify cleaning and transformation requirements before loading the data into PostgreSQL.

In [ ]:
df.isnull().sum() 
duplicates = df.duplicated().sum()

print(f"Duplicate Rows: {duplicates:,}")

# Data Cleaning

Three columns contained approximately 10,000 missing values:

- avg_salary
- revenue_growth
- employee_satisfaction

Missing values were imputed using the median value within each industry group.

Industry-level medians were selected to preserve industry-specific characteristics and avoid distortion caused by global averages.

In [ ]:
df['avg_salary'] = df.groupby('industry')['avg_salary'].transform(lambda x: x.fillna(x.median()))

df['revenue_growth'] = df.groupby('industry')['revenue_growth'].transform(lambda x: x.fillna(x.median()))

df['employee_satisfaction'] = df.groupby('industry')['employee_satisfaction'].transform(lambda x: x.fillna(x.median()))

In [ ]:
# Solution Validation: 

missing_after = df[
    ['avg_salary',
     'revenue_growth',
     'employee_satisfaction']
].isnull().sum()

missing_after

# Exploratory Findings

The distributions of both AI Adoption Score and Automation Level are heavily skewed toward lower values.

Key observations:

- Mean AI Adoption Score: 28.98
- Mean Automation Level: 29.11
- Median value for both metrics: 25
- 75% of organizations have AI Adoption Scores of 42 or lower
- 75% of organizations have Automation Levels of 43 or lower

These findings suggest that highly mature AI implementations and advanced automation remain relatively uncommon within the dataset. Most organizations appear to be in the early stages of AI and automation adoption.


In [ ]:
df['ai_adoption_score'].describe()
df['automation_level'].describe()

In [ ]:
df['ai_adoption_tier'].value_counts()
df['automation_tier'].value_counts()

# Feature Engineering

This section creates derived fields that will support later SQL analysis and dashboard development.

Fields Created: 
- AI Adoption Tier
- Automation Tier

In [ ]:
df['ai_adoption_tier'] = pd.cut(
    df['ai_adoption_score'],
    bins=[0, 25, 50, 75, 100],
    labels=[
        'Minimal',
        'Emerging',
        'Established',
        'Leading'
    ],
    include_lowest=True
)

df['automation_tier'] = pd.cut(
    df['automation_level'],
    bins=[0, 25, 50, 75, 100],
    labels=[
        'Very Low', 
        'Low',
        'Medium',
        'High'
    ],
    include_lowest=True
)

# Next Steps

The cleaned dataset is now ready for loading into PostgreSQL.

Upcoming tasks:

- Design database schema
- Load cleaned data into PostgreSQL
- Perform SQL analysis
- Build Power BI dashboard
- Develop final case study report

In [ ]:
#load cleaned data 

df.to_csv(
    'cleaned_ai_workforce.csv',
    index=False
)

In [ ]:
import sqlalchemy as sal

USER = 'postgres'
PASSWORD = 'admin'
HOST = 'localhost'
PORT = '5432'
DB_NAME = 'ai_workforce'

connection_string = (
    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
)

engine = sal.create_engine(connection_string)

df.to_sql(
    'ai_workforce_data',
    con=engine,
    if_exists='replace',
    index=False
)

In [ ]:
query = """
SELECT COUNT(*)
FROM ai_workforce_data
"""

pd.read_sql(query, engine)